In [1]:
pip install openai pandas tqdm

In [2]:
import pandas as pd
from openai import OpenAI
import os
import time
from tqdm import tqdm
import json
from google.colab import userdata

# Initialize OpenAI client using Colab secrets
client = OpenAI(api_key=userdata.get('OPEN_API'))

# Dataset configuration (following your template structure)
src_lang = "English"
tgt_lang = "Kalenjin"

dev_file = "https://raw.githubusercontent.com/Tonative-Research/Extending-AfriXNLI/refs/heads/main/dev/xnli.dev%20-%20eng_dataset.csv"
test_file = "https://raw.githubusercontent.com/Tonative-Research/Extending-AfriXNLI/refs/heads/main/test/xnli.test%20-%20english.csv"

# Create output directories (following your template)
os.makedirs('dev', exist_ok=True)
os.makedirs('test', exist_ok=True)

sentence1_dev = 'dev/sentence1_translated.csv'
sentence2_dev = 'dev/sentence2_translated.csv'
sentence1_test = 'test/sentence1_translated.csv'
sentence2_test = 'test/sentence2_translated.csv'

# Quota management settings
REQUESTS_PER_MINUTE = 30
DELAY_BETWEEN_REQUESTS = 60 / REQUESTS_PER_MINUTE
CHECKPOINT_EVERY = 10

print(f"Configuration:")
print(f"- Source language: {src_lang}")
print(f"- Target language: {tgt_lang}")
print(f"- Rate limit: {REQUESTS_PER_MINUTE} requests/minute")

Configuration:
- Source language: English
- Target language: Kalenjin
- Rate limit: 30 requests/minute


In [3]:
# CELL 2.5: Google Drive Persistent Storage Setup
from google.colab import drive
import os

# Mount Google Drive for persistent checkpoint storage
drive.mount('/content/drive')

# Create persistent checkpoint directory in Drive
checkpoint_dir = '/content/drive/MyDrive/kalenjin_translation_checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)
print(f"Persistent storage ready: {checkpoint_dir}")

Mounted at /content/drive
Persistent storage ready: /content/drive/MyDrive/kalenjin_translation_checkpoints


In [4]:
df = pd.read_csv(dev_file, sep=",")
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

Dataset shape: (2490, 19)
Columns: ['language', 'gold_label', 'sentence1_binary_parse', 'sentence2_binary_parse', 'sentence1_parse', 'sentence2_parse', 'sentence1', 'sentence2', 'promptID', 'pairID', 'genre', 'label1', 'label2', 'label3', 'label4', 'label5', 'sentence1_tokenized', 'sentence2_tokenized', 'match']


,language,gold_label,sentence1_binary_parse,sentence2_binary_parse,sentence1_parse,sentence2_parse,sentence1,sentence2,promptID,pairID,genre,label1,label2,label3,label4,label5,sentence1_tokenized,sentence2_tokenized,match
0,en,neutral,NaN,NaN,NaN,NaN,"And he said, Mama, I'm home.",He called his mom as soon as the school bus dr...,1,1,facetoface,neutral,contradiction,neutral,neutral,neutral,"And he said , Mama , I 'm home .",He called his mom as soon as the school bus dr...,True
1,en,contradiction,NaN,NaN,NaN,NaN,"And he said, Mama, I'm home.",He didn't say a word.,1,2,facetoface,contradiction,contradiction,contradiction,contradiction,contradiction,"And he said , Mama , I 'm home .",He didn 't say a word .,True
2,en,entailment,NaN,NaN,NaN,NaN,"And he said, Mama, I'm home.",He told his mom he had gotten home.,1,3,facetoface,entailment,entailment,neutral,entailment,entailment,"And he said , Mama , I 'm home .",He told his mom he had gotten home .,True
3,en,neutral,NaN,NaN,NaN,NaN,I didn't know what I was going for or anything...,I have never been to Washington so when I was ...,6,16,facetoface,neutral,neutral,neutral,neutral,neutral,I didn 't know what I was going for or anythin...,I have never been to Washington so when I was ...,True
4,en,contradiction,NaN,NaN,NaN,NaN,I didn't know what I was going for or anything...,I knew exactly what I needed to do as I marche...,6,17,facetoface,contradiction,contradiction,contradiction,contradiction,contradiction,I didn 't know what I was going for or anythin...,I knew exactly what I needed to do as I marche...,True


In [5]:
def translate_text_openai(text, source_lang="English", target_lang="Kalenjin", max_retries=3):
    """
    Translate text using OpenAI API with retry mechanism
    (Modified from your original function for quota safety)
    """
    if pd.isna(text) or text == "":
        return ""

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",  # Using your specified model
                messages=[
                    {
                        "role": "system",
                        "content": f"You are a professional translator. Translate the following text from {source_lang} to {target_lang}. Provide only the translation without any explanations or additional text. Maintain the original meaning and context as accurately as possible."
                    },
                    {
                        "role": "user",
                        "content": f"Translate this text to {target_lang}: {text}"
                    }
                ],
                temperature=0.3,
                max_tokens=512
            )

            translation = response.choices[0].message.content.strip()
            return translation

        except Exception as e:
            print(f"Translation attempt {attempt + 1} failed: {str(e)}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)  # Exponential backoff
            else:
                print(f"Failed to translate: {text}")
                return text  # Return original text if all attempts fail

    return text

# Quota-safe batch translation function
def translate_batch_quota_safe(texts, batch_size=1, delay=None):
    """
    Translate a batch of texts using OpenAI API with quota-safe rate limiting
    (Modified from your batch function)
    """
    if delay is None:
        delay = DELAY_BETWEEN_REQUESTS

    translations = []

    # Progress tracking
    progress_file = f"translation_progress_{int(time.time())}.json"

    for i in tqdm(range(0, len(texts)), desc="Translating"):
        text = texts[i]
        translation = translate_text_openai(str(text) if pd.notna(text) else "")
        translations.append(translation)

        # Rate limiting
        time.sleep(delay)

        # Save progress checkpoint
        if (i + 1) % CHECKPOINT_EVERY == 0:
            with open(progress_file, 'w') as f:
                json.dump({
                    'completed': i + 1,
                    'total': len(texts),
                    'translations': translations
                }, f)
            print(f"Checkpoint saved: {i + 1}/{len(texts)} completed")

    return translations

In [6]:
def process_file(input_path, output_path1, output_path2):
    """
    Process the input file and create translated versions
    """
    print(f"Processing {input_path}...")
    df = pd.read_csv(input_path, sep=",")

    print(f"Dataset shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")


    sentence1_translated_df = pd.DataFrame()
    sentence2_translated_df = pd.DataFrame()

    sentence1_translated_df[['promptID', 'sentence1']] = df[['promptID', 'sentence1']]
    sentence2_translated_df[['promptID', 'sentence2']] = df[['promptID', 'sentence2']]

    # Translate sentences with quota-safe batching
    print("Translating sentence1...")
    sentence1_translated_df["sentence1_translated"] = translate_batch_quota_safe(
        df["sentence1"].tolist()
    )

    print("Translating sentence2...")
    sentence2_translated_df["sentence2_translated"] = translate_batch_quota_safe(
        df["sentence2"].tolist()
    )

    # Reorder columns
    sentence1_translated_df = sentence1_translated_df[["promptID", "sentence1", "sentence1_translated"]]
    sentence2_translated_df = sentence2_translated_df[["promptID", "sentence2", "sentence2_translated"]]

    # Remove duplicates
    sentence1_translated_df = sentence1_translated_df.drop_duplicates()
    sentence2_translated_df = sentence2_translated_df.drop_duplicates()

    # Save files
    sentence1_translated_df.to_csv(output_path1, index=False)
    sentence2_translated_df.to_csv(output_path2, index=False)

    print(f"Saved translated files:")
    print(f"- {output_path1} (shape: {sentence1_translated_df.shape})")
    print(f"- {output_path2} (shape: {sentence2_translated_df.shape})")

In [7]:
def estimate_costs(df):
    """Estimate the cost of translation using OpenAI API (from your template)"""
    total_chars = 0
    for col in ['sentence1', 'sentence2']:
        if col in df.columns:
            total_chars += df[col].astype(str).str.len().sum()

    # Rough estimation: ~4 chars per token, GPT-4o-mini costs $0.15 per 1M input tokens
    estimated_tokens = total_chars / 4
    estimated_cost = (estimated_tokens / 1_000_000) * 0.15

    print(f"Estimated tokens: {estimated_tokens:,.0f}")
    print(f"Estimated cost: ${estimated_cost:.2f}")
    print(f"Estimated time: {len(df) * 2 * DELAY_BETWEEN_REQUESTS / 3600:.1f} hours")
    return estimated_cost

# Estimate costs for the dataset
preview_df = pd.read_csv(dev_file, sep=",")
estimate_costs(preview_df)

Estimated tokens: 98,403
Estimated cost: $0.01
Estimated time: 2.8 hours


np.float64(0.014760412499999999)

In [8]:
# Check for existing checkpoints
checkpoint_files = [f for f in os.listdir('.') if f.endswith('_checkpoint.json')]

if checkpoint_files:
    print(f"Found {len(checkpoint_files)} checkpoint file(s)")
    for file in checkpoint_files:
        try:
            with open(file, 'r') as f:
                data = json.load(f)
            print(f"- {file}: {data.get('last_completed', 0) + 1}/{data.get('total', 0)} completed")
        except Exception as e:
            print(f"- {file}: Error reading checkpoint")
else:
    print("No checkpoint files found. Starting fresh translation.")

No checkpoint files found. Starting fresh translation.


In [9]:
# Convert old checkpoint to new format
import json
import os

# Load the old checkpoint
old_file = "/content/translation_progress_1758624590.json"
if os.path.exists(old_file):
    with open(old_file, 'r') as f:
        old_data = json.load(f)

    # Convert to new format
    new_data = {
        'translations': old_data['translations'],
        'last_completed': old_data['completed'] - 1,  # Convert to 0-based index
        'total': old_data['total'],
        'column_name': 'sentence1_dev',
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
    }

    # Save in new format
    with open('sentence1_dev_checkpoint.json', 'w') as f:
        json.dump(new_data, f, indent=2)

    print(f" Converted checkpoint: Ready to resume from translation {old_data['completed']}!")
    print(f" Progress: {old_data['completed']}/{old_data['total']} ({old_data['completed']/old_data['total']*100:.1f}%)")
else:
    print(" Old checkpoint file not found")

 Old checkpoint file not found


In [10]:
def translate_batch_quota_safe_persistent(texts, column_name, delay=None):
    """Version that saves to Google Drive for persistence across runtime disconnections"""
    if delay is None:
        delay = DELAY_BETWEEN_REQUESTS

    # Save checkpoint to Google Drive (survives runtime disconnections)
    progress_file = os.path.join(checkpoint_dir, f"{column_name}_checkpoint.json")

    translations = [""] * len(texts)
    start_idx = 0

    # load from persistent Drive storage
    if os.path.exists(progress_file):
        try:
            with open(progress_file, 'r') as f:
                checkpoint = json.load(f)

            saved_translations = checkpoint.get('translations', [])
            last_completed = checkpoint.get('last_completed', -1)

            if len(saved_translations) > 0:
                for i, trans in enumerate(saved_translations):
                    if i < len(translations):
                        translations[i] = trans

                start_idx = last_completed + 1
                print(f"RESUMING from Drive: {start_idx}/{len(texts)} translations found")

        except Exception as e:
            print(f"Could not load Drive checkpoint: {e}")

    if start_idx == 0:
        print(f"STARTING fresh: {len(texts)} texts to translate")

    # Translation with frequent Drive saves
    for i in tqdm(range(start_idx, len(texts)), desc=f"Translating {column_name}"):
        text = texts[i]
        translation = translate_text_openai(str(text) if pd.notna(text) else "")
        translations[i] = translation

        time.sleep(delay)

        # Save to Drive every 5 translations (more frequent than before)
        if (i + 1) % 5 == 0:
            checkpoint_data = {
                'translations': translations,
                'last_completed': i,
                'total': len(texts),
                'column_name': column_name,
                'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
            }
            with open(progress_file, 'w') as f:
                json.dump(checkpoint_data, f, ensure_ascii=False)
            print(f"Drive saved: {i + 1}/{len(texts)}")

    # Final save
    final_checkpoint = {
        'translations': translations,
        'last_completed': len(texts) - 1,
        'total': len(texts),
        'column_name': column_name,
        'completed': True,
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
    }
    with open(progress_file, 'w') as f:
        json.dump(final_checkpoint, f, ensure_ascii=False)

    return translations

print("Drive-persistent translation function loaded!")

Drive-persistent translation function loaded!


In [11]:
def process_file_drive_safe(input_path, output_path1, output_path2):
    """Process file with Drive-persistent checkpoints"""
    print(f"Processing {input_path}...")
    df = pd.read_csv(input_path, sep=",")

    sentence1_translated_df = pd.DataFrame()
    sentence2_translated_df = pd.DataFrame()

    sentence1_translated_df[['promptID', 'sentence1']] = df[['promptID', 'sentence1']]
    sentence2_translated_df[['promptID', 'sentence2']] = df[['promptID', 'sentence2']]

    print("Translating sentence1...")
    sentence1_translated_df["sentence1_translated"] = translate_batch_quota_safe_persistent(
        df["sentence1"].tolist(), "sentence1_dev"
    )

    print("Translating sentence2...")
    sentence2_translated_df["sentence2_translated"] = translate_batch_quota_safe_persistent(
        df["sentence2"].tolist(), "sentence2_dev"
    )

    # Save final results to Drive as well
    drive_output1 = os.path.join(checkpoint_dir, "sentence1_final.csv")
    drive_output2 = os.path.join(checkpoint_dir, "sentence2_final.csv")

    sentence1_translated_df.to_csv(output_path1, index=False)
    sentence2_translated_df.to_csv(output_path2, index=False)
    sentence1_translated_df.to_csv(drive_output1, index=False)  # Backup to Drive
    sentence2_translated_df.to_csv(drive_output2, index=False)  # Backup to Drive

    print(f"Results saved to both local and Drive storage")
    return sentence1_translated_df, sentence2_translated_df

In [ ]:
print("\n" + "="*50)
print("PROCESSING DEV SET")
print("="*50)
dev_sent1, dev_sent2 = process_file_fixed(dev_file, sentence1_dev, sentence2_dev)


PROCESSING DEV SET
Processing https://raw.githubusercontent.com/Tonative-Research/Extending-AfriXNLI/refs/heads/main/dev/xnli.dev%20-%20eng_dataset.csv...
Translating sentence1...
 RESUMING from checkpoint: 10/2490 (found 2490 saved translations)
 Skipping first 10 already completed translations


Translating sentence1_dev:   0%|          | 10/2480 [01:12<5:18:32,  7.74s/it]

 Checkpoint saved: 20/2490 completed


Translating sentence1_dev:   1%|          | 20/2480 [02:13<4:08:30,  6.06s/it]

 Checkpoint saved: 30/2490 completed


...

Translating sentence2_dev: 100%|█████████▉| 2480/2490 [2:14:50<00:28,  2.90s/it]

 Checkpoint saved: 2480/2490 completed


Translating sentence2_dev: 100%|██████████| 2490/2490 [2:15:20<00:00,  3.26s/it]

 Checkpoint saved: 2490/2490 completed
 sentence2_dev translation completed!
Saved translated files:
- dev/sentence1_translated.csv (shape: (2474, 3))
- dev/sentence2_translated.csv (shape: (2490, 3))


TypeError: cannot unpack non-iterable NoneType object